# MealMaster RAG Prototype (Buildcamp Homework)

## 1. Title and purpose

This notebook is an educational prototype for AI Buildcamp homework.
It demonstrates a baseline MealMaster RAG pipeline (PDF -> markdown -> cleaning -> chunking -> lexical retrieval -> answer generation).
It is not the production MealMaster retrieval system.

This version is fully standalone and does not depend on the private MealMaster repository.

What it creates:
- source validation/download reports
- extracted markdown files from PDFs
- cleaned document JSON records
- chunk JSONL dataset with rich metadata
- index input JSONL for minsearch rebuild-on-demand

Why this supports future AI work:
- structured outputs can reuse retrieved context
- agent tools can call the same retrieval functions
- eval and monitoring notebooks can reuse saved artifacts without rerunning extraction



## 2. Environment / imports

This cell sets repository paths, checks package availability, and imports helper modules.


In [ ]:
from __future__ import annotations

import importlib.util
from pathlib import Path

import pandas as pd

# Repo-agnostic: use current working directory (not private repo paths).
REPO_ROOT = Path.cwd().resolve()

required_packages = ['markitdown', 'minsearch', 'pydantic', 'pandas']
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
optional_packages = ['openai', 'tiktoken']
missing_optional = [name for name in optional_packages if importlib.util.find_spec(name) is None]

print('Working directory:', REPO_ROOT)
print('Missing required packages:', missing_packages or 'none')
print('Missing optional packages (needed for live token counting + LLM demo):', missing_optional or 'none')

if missing_packages:
    raise ModuleNotFoundError(
        "Install missing required packages in your notebook environment, then rerun. "
        f"Missing: {missing_packages}. "
        "Suggested: uv add 'markitdown[pdf]' minsearch pydantic pandas openai tiktoken"
    )



In [ ]:
import json
import os
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal
from urllib.parse import urlparse
from urllib.request import Request, urlopen

from markitdown import MarkItDown
from minsearch import Index
from pydantic import BaseModel, ConfigDict, Field

try:
    import tiktoken
except Exception:
    tiktoken = None

try:
    from openai import OpenAI
except Exception:
    OpenAI = None


@dataclass
class RAGConfig:
    repo_root: Path
    data_root: Path
    raw_root: Path
    extracted_root: Path
    cleaned_root: Path
    chunks_root: Path
    indexes_root: Path
    manifests_root: Path
    reports_root: Path
    llm_model: str = 'gpt-4o-mini'


@dataclass
class SourceManifestEntry:
    doc_id: str
    title: str = ''
    source_org: str = ''
    source_url: str = ''
    source_path: str = ''
    filename: str = ''
    collection: str = 'healthy_aging_general'
    audience: str = 'general_adults'
    topic_tags: list[str] = field(default_factory=list)
    guidance_type: str = 'general_guidance'
    stance: str = 'general_public_health'
    language: str = 'en'
    enabled: bool = True
    download_now: bool = False
    notes: str = ''


@dataclass
class CleanedDocument:
    doc_id: str
    title: str
    source_url: str
    collection: str
    audience: str
    content_lines: list[str]
    line_count: int
    char_count: int


@dataclass
class ChunkRecord:
    chunk_id: str
    doc_id: str
    source_title: str
    source_url: str
    collection: str
    line_start: int
    line_end: int
    line_count: int
    char_count: int
    content: str


class StructuredRAGResponse(BaseModel):
    model_config = ConfigDict(extra='forbid')

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description='True if relevant information was found in retrieval context')
    confidence: float = Field(description='Confidence score from 0.0 to 1.0')
    confidence_explanation: str = Field(description='Explanation for confidence value')
    answer_type: Literal['how-to', 'explanation', 'troubleshooting', 'comparison', 'reference'] = Field(description='Answer category')
    followup_questions: list[str] = Field(description='Suggested follow-up questions')


def default_config(repo_root: Path) -> RAGConfig:
    data_root = repo_root / 'data' / 'mealmaster_rag'
    return RAGConfig(
        repo_root=repo_root,
        data_root=data_root,
        raw_root=data_root / 'raw',
        extracted_root=data_root / 'extracted_markdown',
        cleaned_root=data_root / 'cleaned_documents',
        chunks_root=data_root / 'chunks',
        indexes_root=data_root / 'indexes',
        manifests_root=data_root / 'manifests',
        reports_root=data_root / 'reports',
    )


def ensure_directories(config: RAGConfig) -> None:
    for path in [
        config.data_root,
        config.raw_root,
        config.extracted_root,
        config.cleaned_root,
        config.chunks_root,
        config.indexes_root,
        config.manifests_root,
        config.reports_root,
    ]:
        path.mkdir(parents=True, exist_ok=True)


def _default_seed_rows() -> list[dict[str, Any]]:
    return [
        {
            'doc_id': 'eating4health',
            'title': 'Healthy Eating for Your Brain and Body',
            'source_org': 'The Ohio State University Wexner Medical Center',
            'source_url': 'https://healthsystem.osumc.edu/pteduc/docs/Eating4Health.pdf',
            'filename': 'Eating4Health.pdf',
            'collection': 'brain_mood_sleep',
            'audience': 'general_adults',
        },
        {
            'doc_id': 'nia_make_smart_food_choices',
            'title': 'Make Smart Food Choices for Healthy Aging',
            'source_org': 'National Institute on Aging',
            'source_url': 'https://www.nia.nih.gov/sites/default/files/makesmartfoodchoices-508.pdf',
            'filename': 'makesmartfoodchoices-508.pdf',
            'collection': 'healthy_aging_general',
            'audience': 'older_adults',
        },
        {
            'doc_id': 'care_connection_eating_right',
            'title': 'Eating Right for Older Adults',
            'source_org': 'Care Connection for Aging Services',
            'source_url': 'https://goaging.org/wp-content/uploads/2024/03/NNM_Eating-Right-Tips-for-Older-Adults_English.pdf',
            'filename': 'NNM_Eating-Right-Tips-for-Older-Adults_English.pdf',
            'collection': 'healthy_aging_general',
            'audience': 'older_adults',
        },
        {
            'doc_id': 'nia_healthy_eating_routine',
            'title': 'Build a Healthy Eating Routine as You Get Older',
            'source_org': 'National Institute on Aging',
            'source_url': 'https://www.nia.nih.gov/sites/default/files/2023-12/healthy-eating-routine-as-you-get-older.pdf',
            'filename': 'healthy-eating-routine-as-you-get-older.pdf',
            'collection': 'healthy_aging_general',
            'audience': 'older_adults',
        },
        {
            'doc_id': 'eat_well_for_life',
            'title': 'Eat Well for Life',
            'source_org': 'American Heart Association',
            'source_url': 'https://www.heart.org/-/media/Files/Healthy-Living/Healthy-Eating/Eat-Smart/Nutrition-Basics/Eat-Well-for-Life.pdf',
            'filename': 'Eat-Well-for-Life.pdf',
            'collection': 'healthy_aging_general',
            'audience': 'general_adults',
        },
    ]


def _entry_from_row(row: dict[str, Any]) -> SourceManifestEntry:
    doc_id = str(row.get('doc_id', '')).strip()
    filename = str(row.get('filename', '')).strip()
    source_url = str(row.get('source_url', '')).strip()
    if not filename and source_url:
        filename = Path(urlparse(source_url).path).name
    if not filename:
        filename = f'{doc_id}.pdf'

    collection = str(row.get('collection', 'healthy_aging_general')).strip() or 'healthy_aging_general'
    source_path = str((Path('data/mealmaster_rag/raw') / collection / filename).as_posix())

    return SourceManifestEntry(
        doc_id=doc_id,
        title=str(row.get('title', '')).strip(),
        source_org=str(row.get('source_org', '')).strip(),
        source_url=source_url,
        source_path=source_path,
        filename=filename,
        collection=collection,
        audience=str(row.get('audience', 'general_adults')).strip() or 'general_adults',
        topic_tags=list(row.get('topic_tags', []) or []),
        guidance_type=str(row.get('guidance_type', 'general_guidance')).strip() or 'general_guidance',
        stance=str(row.get('stance', 'general_public_health')).strip() or 'general_public_health',
        language=str(row.get('language', 'en')).strip() or 'en',
        enabled=bool(row.get('enabled', True)),
        download_now=bool(row.get('download_now', False)),
        notes=str(row.get('notes', '')).strip(),
    )


def entries_to_rows(entries: list[SourceManifestEntry]) -> list[dict[str, Any]]:
    return [asdict(entry) for entry in entries]


def load_manifest(path: Path) -> list[SourceManifestEntry]:
    if not path.exists():
        return []
    raw = json.loads(path.read_text(encoding='utf-8'))
    return [SourceManifestEntry(**row) for row in raw]


def save_manifest(entries: list[SourceManifestEntry], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = [asdict(entry) for entry in entries]
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


def ensure_seed_manifest(path: Path, repo_root: Path, overwrite: bool = False) -> list[SourceManifestEntry]:
    if path.exists() and not overwrite:
        return load_manifest(path)

    seed_entries = [_entry_from_row(row) for row in _default_seed_rows()]
    save_manifest(seed_entries, path)
    return seed_entries


def validate_sources(
    entries: list[SourceManifestEntry],
    repo_root: Path,
    allow_public_pdf_downloads: bool,
    allow_archive_auto_download: bool,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for entry in entries:
        abs_path = repo_root / entry.source_path
        source_url = (entry.source_url or '').strip()

        if not entry.enabled:
            status = 'disabled'
        elif abs_path.exists() and abs_path.stat().st_size > 0:
            status = 'ready_local'
        elif source_url and source_url.lower().endswith('.pdf'):
            if 'archive.org' in source_url.lower() and not allow_archive_auto_download:
                status = 'archive_download_disabled'
            elif allow_public_pdf_downloads:
                status = 'downloadable_public_pdf'
            else:
                status = 'missing_local_download_disabled'
        elif source_url:
            status = 'non_pdf_or_unsupported_url'
        else:
            status = 'missing_source'

        rows.append(
            {
                'doc_id': entry.doc_id,
                'title': entry.title,
                'status': status,
                'source_path': entry.source_path,
                'absolute_source_path': str(abs_path),
                'source_url': source_url,
                'error': '',
            }
        )
    return rows


def download_missing_pdfs(validation_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    results = []
    for row in validation_rows:
        if row.get('status') != 'downloadable_public_pdf':
            continue

        target = Path(str(row['absolute_source_path']))
        target.parent.mkdir(parents=True, exist_ok=True)

        try:
            req = Request(row['source_url'], headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(req, timeout=60) as resp:
                data = resp.read()

            if not data:
                raise RuntimeError('empty response body')
            if not str(target).lower().endswith('.pdf'):
                raise RuntimeError('target does not end with .pdf')

            target.write_bytes(data)
            results.append(
                {
                    'doc_id': row['doc_id'],
                    'status': 'downloaded',
                    'source_path': row['source_path'],
                    'error': '',
                }
            )
        except Exception as e:
            results.append(
                {
                    'doc_id': row['doc_id'],
                    'status': 'failed',
                    'source_path': row['source_path'],
                    'error': str(e),
                }
            )
    return results


def extract_markdown(
    entries: list[SourceManifestEntry],
    repo_root: Path,
    extracted_root: Path,
    force_rebuild: bool,
) -> list[dict[str, Any]]:
    extracted_root.mkdir(parents=True, exist_ok=True)
    converter = MarkItDown()
    rows: list[dict[str, Any]] = []

    for entry in entries:
        source_path = repo_root / entry.source_path
        output_path = extracted_root / f'{entry.doc_id}.md'

        if not entry.enabled:
            rows.append({'doc_id': entry.doc_id, 'status': 'disabled', 'source_path': entry.source_path, 'output_path': str(output_path), 'error': ''})
            continue

        if not source_path.exists():
            rows.append({'doc_id': entry.doc_id, 'status': 'missing_source_pdf', 'source_path': entry.source_path, 'output_path': str(output_path), 'error': 'source PDF not found'})
            continue

        if output_path.exists() and not force_rebuild:
            rows.append({'doc_id': entry.doc_id, 'status': 'skipped_existing', 'source_path': entry.source_path, 'output_path': str(output_path), 'error': ''})
            continue

        try:
            result = converter.convert(str(source_path))
            markdown = (result.markdown or result.text_content or '').strip()
            if not markdown:
                raise RuntimeError('empty markdown output')
            output_path.write_text(markdown, encoding='utf-8')
            rows.append({'doc_id': entry.doc_id, 'status': 'extracted', 'source_path': entry.source_path, 'output_path': str(output_path), 'error': ''})
        except Exception as e:
            rows.append({'doc_id': entry.doc_id, 'status': 'failed', 'source_path': entry.source_path, 'output_path': str(output_path), 'error': str(e)})

    return rows


def prepare_cleaned_documents(
    entries: list[SourceManifestEntry],
    repo_root: Path,
    extracted_root: Path,
    cleaned_root: Path,
    force_rebuild: bool,
    normalize_whitespace: bool = True,
) -> tuple[list[CleanedDocument], list[dict[str, Any]]]:
    cleaned_root.mkdir(parents=True, exist_ok=True)
    docs: list[CleanedDocument] = []
    rows: list[dict[str, Any]] = []

    for entry in entries:
        md_path = extracted_root / f'{entry.doc_id}.md'
        clean_path = cleaned_root / f'{entry.doc_id}.json'

        if not entry.enabled:
            rows.append({'doc_id': entry.doc_id, 'status': 'disabled', 'output_path': str(clean_path), 'error': ''})
            continue

        if not md_path.exists():
            rows.append({'doc_id': entry.doc_id, 'status': 'missing_extracted_markdown', 'output_path': str(clean_path), 'error': 'markdown file missing'})
            continue

        text = md_path.read_text(encoding='utf-8', errors='ignore')
        lines = []
        for line in text.splitlines():
            value = line.strip() if normalize_whitespace else line
            if value:
                lines.append(value)

        doc = CleanedDocument(
            doc_id=entry.doc_id,
            title=entry.title,
            source_url=entry.source_url,
            collection=entry.collection,
            audience=entry.audience,
            content_lines=lines,
            line_count=len(lines),
            char_count=sum(len(x) for x in lines),
        )
        docs.append(doc)

        payload = asdict(doc)
        clean_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        rows.append({'doc_id': entry.doc_id, 'status': 'cleaned', 'output_path': str(clean_path), 'error': ''})

    return docs, rows


def save_cleaned_documents_jsonl(cleaned_docs: list[CleanedDocument], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for doc in cleaned_docs:
            f.write(json.dumps(asdict(doc), ensure_ascii=False) + '\n')


def build_chunks(cleaned_docs: list[CleanedDocument], size: int, step: int) -> list[ChunkRecord]:
    chunks: list[ChunkRecord] = []
    for doc in cleaned_docs:
        lines = doc.content_lines
        if not lines:
            continue

        start_positions = [0] if len(lines) <= size else list(range(0, len(lines), step))
        part = 0
        for start in start_positions:
            window = lines[start:start + size]
            if not window:
                continue
            part += 1
            content = '\n'.join(window)
            chunks.append(
                ChunkRecord(
                    chunk_id=f'{doc.doc_id}-chunk-{part}',
                    doc_id=doc.doc_id,
                    source_title=doc.title,
                    source_url=doc.source_url,
                    collection=doc.collection,
                    line_start=start,
                    line_end=start + len(window),
                    line_count=len(window),
                    char_count=len(content),
                    content=content,
                )
            )
            if len(lines) <= size:
                break
    return chunks


def save_chunks_jsonl(chunks: list[ChunkRecord], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for chunk in chunks:
            f.write(json.dumps(asdict(chunk), ensure_ascii=False) + '\n')


def prepare_index_documents(chunks: list[ChunkRecord]) -> list[dict[str, Any]]:
    return [
        {
            'id': chunk.chunk_id,
            'doc_id': chunk.doc_id,
            'source_title': chunk.source_title,
            'source_url': chunk.source_url,
            'collection': chunk.collection,
            'line_start': chunk.line_start,
            'line_end': chunk.line_end,
            'content': chunk.content,
        }
        for chunk in chunks
    ]


def build_index(index_documents: list[dict[str, Any]]) -> Index:
    index = Index(
        text_fields=['content', 'source_title'],
        keyword_fields=['id', 'doc_id', 'collection', 'source_url'],
    )
    if index_documents:
        index.fit(index_documents)
    return index


def save_index_documents(index_documents: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for doc in index_documents:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')


def run_queries(index: Index, queries: list[str], num_results: int) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for query in queries:
        results = index.search(query, num_results=num_results) if index else []
        for rank, item in enumerate(results, start=1):
            rows.append(
                {
                    'query': query,
                    'rank': rank,
                    'source_title': item.get('source_title', ''),
                    'source_url': item.get('source_url', ''),
                    'collection': item.get('collection', ''),
                    'snippet': str(item.get('content', ''))[:260],
                }
            )
    return rows


def _estimate_tokens(text: str, model: str) -> int:
    if tiktoken is not None:
        try:
            enc = tiktoken.encoding_for_model(model)
            return len(enc.encode(text))
        except Exception:
            pass
    return max(1, len(text) // 4)


def _build_rag_prompt(query: str, results: list[dict[str, Any]]) -> str:
    slim = [
        {
            'source_title': r.get('source_title', ''),
            'source_url': r.get('source_url', ''),
            'collection': r.get('collection', ''),
            'content': r.get('content', ''),
        }
        for r in results
    ]
    return (
        '<QUESTION>\n'
        + query
        + '\n</QUESTION>\n\n<CONTEXT>\n'
        + json.dumps(slim, ensure_ascii=False, indent=2)
        + '\n</CONTEXT>'
    )


def _openai_client() -> Any | None:
    if OpenAI is None:
        return None
    if not os.environ.get('OPENAI_API_KEY'):
        return None
    return OpenAI()


def rag_answer(index: Index, query: str, model: str, num_results: int) -> dict[str, Any]:
    references = index.search(query, num_results=num_results) if index else []
    prompt = _build_rag_prompt(query, references)
    instructions = (
        "You're a nutrition assistant. Answer the QUESTION using only the provided CONTEXT. "
        'If evidence is insufficient, say so clearly.'
    )

    client = _openai_client()
    if client is None:
        return {
            'mode': 'estimated',
            'question': query,
            'answer': None,
            'prompt_preview': prompt,
            'references': references,
            'usage': {
                'input_tokens': _estimate_tokens(instructions + '\n' + prompt, model),
                'output_tokens': None,
            },
        }

    response = client.responses.create(
        model=model,
        input=[
            {'role': 'system', 'content': instructions},
            {'role': 'user', 'content': prompt},
        ],
    )
    usage = response.usage
    return {
        'mode': 'llm',
        'question': query,
        'answer': response.output_text,
        'prompt_preview': prompt,
        'references': references,
        'usage': {
            'input_tokens': int(getattr(usage, 'input_tokens', 0) or 0),
            'output_tokens': int(getattr(usage, 'output_tokens', 0) or 0),
        },
    }


def rag_answer_structured(index: Index, query: str, model: str, num_results: int) -> dict[str, Any]:
    references = index.search(query, num_results=num_results) if index else []
    prompt = _build_rag_prompt(query, references)
    instructions = (
        "You're a nutrition assistant. Produce structured JSON based only on CONTEXT; "
        'if uncertain, lower confidence and explain why.'
    )

    client = _openai_client()
    if client is None:
        return {
            'mode': 'estimated',
            'usage': {
                'input_tokens': _estimate_tokens(instructions + '\n' + prompt + json.dumps(StructuredRAGResponse.model_json_schema()), model),
                'output_tokens': None,
            },
            'response': {
                'answer': 'OPENAI_API_KEY not set. Configure it to run live structured RAG.',
                'found_answer': False,
                'confidence': 0.0,
                'confidence_explanation': 'No live model call in fallback mode.',
                'answer_type': 'reference',
                'followup_questions': [],
            },
            'references': references,
        }

    response = client.responses.create(
        model=model,
        input=[
            {'role': 'system', 'content': instructions},
            {'role': 'user', 'content': prompt},
        ],
        text={
            'format': {
                'type': 'json_schema',
                'name': 'rag_response',
                'schema': StructuredRAGResponse.model_json_schema(),
                'strict': True,
            }
        },
    )

    usage = response.usage
    parsed = StructuredRAGResponse.model_validate_json(response.output_text).model_dump()
    return {
        'mode': 'llm',
        'usage': {
            'input_tokens': int(getattr(usage, 'input_tokens', 0) or 0),
            'output_tokens': int(getattr(usage, 'output_tokens', 0) or 0),
        },
        'response': parsed,
        'references': references,
    }



## 3. Config

All paths and key settings are centralized here.


In [ ]:
config = default_config(REPO_ROOT)
ensure_directories(config)

ALLOW_PUBLIC_PDF_DOWNLOADS = False
ALLOW_ARCHIVE_AUTO_DOWNLOAD = False
FORCE_REBUILD = False

CHUNK_SIZE = 100
CHUNK_STEP = 50
NUM_RESULTS = 5
LLM_MODEL = config.llm_model

MANIFEST_PATH = config.manifests_root / 'seed_sources.json'
VALIDATION_REPORT_PATH = config.reports_root / 'source_validation_report.csv'
DOWNLOAD_REPORT_PATH = config.reports_root / 'source_download_report.csv'
EXTRACTION_REPORT_PATH = config.reports_root / 'extraction_report.csv'
CLEAN_REPORT_PATH = config.reports_root / 'clean_report.csv'
CORPUS_DIAGNOSTICS_PATH = config.reports_root / 'corpus_diagnostics.csv'
CHUNK_DIAGNOSTICS_PATH = config.reports_root / 'chunk_diagnostics.csv'
SEARCH_DEMO_REPORT_PATH = config.reports_root / 'search_demo_results.csv'

CLEANED_DOCS_JSONL_PATH = config.cleaned_root / 'cleaned_documents.jsonl'
CHUNKS_JSONL_PATH = config.chunks_root / 'mealmaster_chunks.jsonl'
INDEX_DOCUMENTS_JSONL_PATH = config.indexes_root / 'index_documents.jsonl'
ARTIFACT_SUMMARY_PATH = config.reports_root / 'artifact_summary.csv'

print('Data root:', config.data_root)
print('Manifest path:', MANIFEST_PATH)


## 3A. Manual source entry and PDF download

This section lets you extend the MealMaster RAG corpus by pasting public PDF URLs directly in the notebook.

Workflow in this section:
1. edit `MANUAL_SOURCE_ROWS`
2. set download controls
3. run manual download (only `enabled=True` and `download_now=True` rows)
4. optionally merge rows into the persisted manifest

This is explicit/manual by design: no silent bulk download.


## Paste new PDF URLs here

Duplicate existing rows in the next editable list, change `source_url`, set `download_now=True`, and rerun manual download cells.
This section is designed so you can extend the RAG corpus without editing helper modules.


In [ ]:
# User-editable list of source rows for manual intake.
# Pre-filled with current public PDF sources used in this project.
MANUAL_SOURCE_ROWS = [
    {
        "doc_id": "eating4health",
        "title": "Healthy Eating for Your Brain and Body",
        "source_org": "The Ohio State University Wexner Medical Center",
        "source_url": "https://healthsystem.osumc.edu/pteduc/docs/Eating4Health.pdf",
        "filename": "Eating4Health.pdf",
        "collection": "brain_mood_sleep",
        "audience": "general_adults",
        "topic_tags": ["healthy_eating", "mood", "brain_health", "meal_patterns", "snacks", "substitutions"],
        "guidance_type": "general_guidance",
        "stance": "general_public_health",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    {
        "doc_id": "nia_make_smart_food_choices",
        "title": "Make Smart Food Choices for Healthy Aging",
        "source_org": "National Institute on Aging",
        "source_url": "https://www.nia.nih.gov/sites/default/files/makesmartfoodchoices-508.pdf",
        "filename": "makesmartfoodchoices-508.pdf",
        "collection": "healthy_aging_general",
        "audience": "older_adults",
        "topic_tags": ["healthy_aging", "food_groups", "weight", "nutrients", "healthy_routine"],
        "guidance_type": "general_guidance",
        "stance": "general_public_health",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    {
        "doc_id": "care_connection_eating_right",
        "title": "Eating Right for Older Adults",
        "source_org": "Care Connection for Aging Services",
        "source_url": "https://goaging.org/wp-content/uploads/2024/03/NNM_Eating-Right-Tips-for-Older-Adults_English.pdf",
        "filename": "NNM_Eating-Right-Tips-for-Older-Adults_English.pdf",
        "collection": "healthy_aging_general",
        "audience": "older_adults",
        "topic_tags": ["healthy_aging", "protein", "hydration", "calcium", "vitamin_d", "meal_balance"],
        "guidance_type": "general_guidance",
        "stance": "general_public_health",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    {
        "doc_id": "odphp_build_healthy_routine",
        "title": "Build a Healthy Eating Routine as You Get Older",
        "source_org": "Office of Disease Prevention and Health Promotion",
        "source_url": "https://odphp.health.gov/sites/default/files/2021-12/DGA_OlderAdults_FactSheet_508cFinal.pdf",
        "filename": "DGA_OlderAdults_FactSheet_508cFinal.pdf",
        "collection": "healthy_aging_general",
        "audience": "older_adults",
        "topic_tags": ["healthy_aging", "protein", "b12", "hydration", "food_safety", "labels"],
        "guidance_type": "general_guidance",
        "stance": "general_public_health",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    {
        "doc_id": "downstate_eating_health_longevity",
        "title": "Eating for Health and Longevity",
        "source_org": "SUNY Downstate Health Sciences University",
        "source_url": "https://www.downstate.edu/about/community-impact/plant-based/_documents/eating_for_health_longevity.pdf",
        "filename": "eating_for_health_longevity.pdf",
        "collection": "plant_based_longevity",
        "audience": "general_adults",
        "topic_tags": ["plant_based", "longevity", "whole_foods", "substitutions", "budget_friendly"],
        "guidance_type": "general_guidance",
        "stance": "plant_based_viewpoint",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    {
        "doc_id": "nsw_eat_well_for_life",
        "title": "Eat Well for Life - Nutrition for the well older person",
        "source_org": "Central Coast Local Health District - NSW Health",
        "source_url": "https://www.cclhd.health.nsw.gov.au/wp-content/uploads/Eat-Well-for-Life.-Nutrition-for-the-well-Older-Person.pdf",
        "filename": "Eat-Well-for-Life.-Nutrition-for-the-well-Older-Person.pdf",
        "collection": "healthy_aging_general",
        "audience": "older_adults",
        "topic_tags": ["healthy_aging", "protein", "calcium", "vitamin_d", "hydration", "low_gi", "food_safety"],
        "guidance_type": "general_guidance",
        "stance": "general_public_health",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    {
        "doc_id": "uw_tips_healthy_aging",
        "title": "Tips for Healthy Eating & Healthy Aging",
        "source_org": "University of Washington",
        "source_url": "https://agerrtc.washington.edu/sites/agerrtc/files/files/Nutrition_Final_20171218(1).pdf",
        "filename": "Nutrition_Final_20171218(1).pdf",
        "collection": "healthy_aging_general",
        "audience": "older_adults",
        "topic_tags": ["healthy_aging", "healthy_eating", "processed_foods", "hydration", "daily_habits"],
        "guidance_type": "general_guidance",
        "stance": "general_public_health",
        "language": "en",
        "enabled": True,
        "download_now": False,
        "notes": "",
    },
    # Example: new public PDF
    # {
    #     "doc_id": "new_public_pdf_doc",
    #     "title": "New Public PDF",
    #     "source_org": "Org Name",
    #     "source_url": "https://example.org/new-file.pdf",
    #     "filename": "new-file.pdf",
    #     "collection": "healthy_aging_general",
    #     "audience": "general_adults",
    #     "topic_tags": ["healthy_eating"],
    #     "guidance_type": "general_guidance",
    #     "stance": "general_public_health",
    #     "language": "en",
    #     "enabled": True,
    #     "download_now": False,
    #     "notes": "",
    # },
    # Example: local-only source (leave source_url blank)
    # {
    #     "doc_id": "new_local_only_doc",
    #     "title": "Local-only PDF",
    #     "source_org": "Local Upload",
    #     "source_url": "",
    #     "filename": "local-only.pdf",
    #     "collection": "plant_based_longevity",
    #     "audience": "general_adults",
    #     "topic_tags": ["plant_based"],
    #     "guidance_type": "general_guidance",
    #     "stance": "general_public_health",
    #     "language": "en",
    #     "enabled": True,
    #     "download_now": False,
    #     "notes": "Place file manually under raw/<collection>/",
    # },
    # Example: disabled placeholder
    # {
    #     "doc_id": "new_disabled_placeholder",
    #     "title": "Archive Placeholder",
    #     "source_org": "Internet Archive",
    #     "source_url": "https://archive.org/details/some-item",
    #     "filename": "some-item",
    #     "collection": "optional_natural_healing_reference",
    #     "audience": "general_adults",
    #     "topic_tags": ["reference"],
    #     "guidance_type": "reference",
    #     "stance": "archive_placeholder",
    #     "language": "en",
    #     "enabled": False,
    #     "download_now": False,
    #     "notes": "Do not auto-download from Archive item pages.",
    # },
]

manual_source_df = pd.DataFrame(MANUAL_SOURCE_ROWS)
manual_source_df


In [ ]:
DOWNLOAD_SELECTED_SOURCES = True
OVERWRITE_EXISTING_DOWNLOADS = False
STRICT_PDF_CHECK = True

print('DOWNLOAD_SELECTED_SOURCES:', DOWNLOAD_SELECTED_SOURCES)
print('OVERWRITE_EXISTING_DOWNLOADS:', OVERWRITE_EXISTING_DOWNLOADS)
print('STRICT_PDF_CHECK:', STRICT_PDF_CHECK)


In [ ]:
from urllib.parse import urlparse
import subprocess

MANUAL_DOWNLOAD_REPORT_PATH = config.reports_root / 'manual_source_download_report.csv'


def _is_archive_url(url: str) -> bool:
    return 'archive.org' in (url or '').lower()


def _validate_downloadable_pdf_url(url: str, strict_pdf_check: bool) -> tuple[bool, str]:
    value = (url or '').strip()
    if not value:
        return False, 'missing_source_url'
    if _is_archive_url(value):
        return False, 'archive_placeholder_or_item_page'

    parsed = urlparse(value)
    if parsed.scheme not in {'http', 'https'}:
        return False, 'non_http_url'

    if strict_pdf_check and not parsed.path.lower().endswith('.pdf'):
        return False, 'non_pdf_url'

    return True, 'ok'


manual_download_results = []

for row in MANUAL_SOURCE_ROWS:
    doc_id = str(row.get('doc_id', '')).strip()
    title = str(row.get('title', '')).strip()
    enabled = bool(row.get('enabled', True))
    download_now = bool(row.get('download_now', False))
    source_url = str(row.get('source_url', '')).strip()

    collection = str(row.get('collection', 'healthy_aging_general')).strip() or 'healthy_aging_general'
    filename = str(row.get('filename', '')).strip()
    if not filename and source_url:
        filename = Path(urlparse(source_url).path).name
    if not filename:
        filename = f'{doc_id or "unknown"}.pdf'

    target_path = config.raw_root / collection / filename
    target_path.parent.mkdir(parents=True, exist_ok=True)

    status = 'skipped'
    reason = ''
    error = ''

    if not enabled:
        reason = 'disabled'
    elif not DOWNLOAD_SELECTED_SOURCES:
        reason = 'global_download_toggle_off'
    elif not download_now:
        reason = 'download_now_false'
    else:
        is_valid, reason = _validate_downloadable_pdf_url(source_url, STRICT_PDF_CHECK)
        if is_valid:
            if target_path.exists() and not OVERWRITE_EXISTING_DOWNLOADS:
                status = 'already_present'
                reason = 'file_exists'
            else:
                try:
                    subprocess.run(
                        [
                            'curl',
                            '-fL',
                            '--retry',
                            '3',
                            '--retry-delay',
                            '2',
                            source_url,
                            '-o',
                            str(target_path),
                        ],
                        check=True,
                        capture_output=True,
                        text=True,
                    )
                    status = 'downloaded'
                    reason = 'download_ok'
                except subprocess.CalledProcessError as exc:
                    status = 'failed'
                    reason = 'curl_error'
                    error = (exc.stderr or exc.stdout or str(exc)).strip()

    if status == 'skipped' and not reason:
        reason = 'not_selected'

    manual_download_results.append(
        {
            'doc_id': doc_id,
            'title': title,
            'enabled': enabled,
            'download_now': download_now,
            'source_url': source_url,
            'target_path': str(target_path),
            'status': status,
            'reason': reason,
            'error': error,
        }
    )

manual_download_df = pd.DataFrame(manual_download_results)
manual_download_df.to_csv(MANUAL_DOWNLOAD_REPORT_PATH, index=False)

print('Saved manual download report:', MANUAL_DOWNLOAD_REPORT_PATH)
manual_download_df


In [ ]:
UPDATE_EXISTING_MANIFEST_ROWS = False
MERGE_MANUAL_ROWS_INTO_MANIFEST = False

existing_entries = load_manifest(MANIFEST_PATH) if MANIFEST_PATH.exists() else []
entries_by_doc_id = {entry.doc_id: entry for entry in existing_entries}
merge_rows = []

for row in MANUAL_SOURCE_ROWS:
    doc_id = str(row.get('doc_id', '')).strip()
    if not doc_id:
        merge_rows.append({'doc_id': '', 'action': 'skipped_missing_doc_id'})
        continue

    collection = str(row.get('collection', 'healthy_aging_general')).strip() or 'healthy_aging_general'
    filename = str(row.get('filename', '')).strip()
    source_url = str(row.get('source_url', '')).strip()

    if not filename and source_url:
        from urllib.parse import urlparse
        filename = Path(urlparse(source_url).path).name
    if not filename:
        filename = f'{doc_id}.pdf'

    source_path = str((Path('data/mealmaster_rag/raw') / collection / filename).as_posix())
    source_url_value = source_url if source_url else f'local://{source_path}'

    entry_payload = {
        'doc_id': doc_id,
        'title': str(row.get('title', '')).strip(),
        'source_org': str(row.get('source_org', '')).strip(),
        'source_url': source_url_value,
        'filename': filename,
        'source_path': source_path,
        'collection': collection,
        'audience': str(row.get('audience', 'general_adults')).strip() or 'general_adults',
        'topic_tags': row.get('topic_tags', []),
        'guidance_type': str(row.get('guidance_type', 'general_guidance')).strip() or 'general_guidance',
        'stance': str(row.get('stance', 'general_public_health')).strip() or 'general_public_health',
        'language': str(row.get('language', 'en')).strip() or 'en',
        'enabled': bool(row.get('enabled', True)),
        'notes': str(row.get('notes', '')).strip() or None,
    }

    topic_tags = entry_payload['topic_tags']
    if isinstance(topic_tags, str):
        entry_payload['topic_tags'] = [tag.strip() for tag in topic_tags.split('|') if tag.strip()]

    exists = doc_id in entries_by_doc_id
    if exists and not UPDATE_EXISTING_MANIFEST_ROWS:
        merge_rows.append({'doc_id': doc_id, 'action': 'skipped_existing'})
        continue

    try:
        validated_entry = SourceManifestEntry.model_validate(entry_payload)
    except Exception as exc:  # noqa: BLE001
        merge_rows.append({'doc_id': doc_id, 'action': 'failed_validation', 'error': str(exc)})
        continue

    entries_by_doc_id[doc_id] = validated_entry
    merge_rows.append({'doc_id': doc_id, 'action': 'updated' if exists else 'added'})

if MERGE_MANUAL_ROWS_INTO_MANIFEST:
    save_manifest(list(entries_by_doc_id.values()), MANIFEST_PATH)
    print('Manifest saved:', MANIFEST_PATH)
else:
    print('Dry run: MERGE_MANUAL_ROWS_INTO_MANIFEST is False. No manifest write performed.')

manual_merge_df = pd.DataFrame(merge_rows)
manual_merge_df


After manual download and optional manifest merge, continue with the next sections (manifest loading, validation, extraction, cleaning, chunking, indexing).


## 4. Corpus manifest loading

The manifest is metadata-first and drives routing (collection, audience, tags), download policy, and processing.


In [ ]:
entries = ensure_seed_manifest(MANIFEST_PATH, REPO_ROOT, overwrite=False)
manifest_df = pd.DataFrame(entries_to_rows(entries))
manifest_df


## 5. Input validation

Rules:
- local files are preferred
- enabled public PDF URLs are marked downloadable if missing
- disabled entries are skipped
- Archive placeholders are not auto-downloaded
- processing can continue with partial availability


In [ ]:
validation_rows = validate_sources(
    entries=entries,
    repo_root=REPO_ROOT,
    allow_public_pdf_downloads=ALLOW_PUBLIC_PDF_DOWNLOADS,
    allow_archive_auto_download=ALLOW_ARCHIVE_AUTO_DOWNLOAD,
)
validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(VALIDATION_REPORT_PATH, index=False)

status_counts = validation_df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count')
print('Validation summary:')
display(status_counts)
validation_df


## 6. Optional public PDF download step

Downloads only rows marked `downloadable`.
Archive placeholders remain skipped.


In [ ]:
if ALLOW_PUBLIC_PDF_DOWNLOADS:
    download_rows = download_missing_pdfs(validation_rows)
else:
    download_rows = []

download_df = pd.DataFrame(download_rows)
if not download_df.empty:
    download_df.to_csv(DOWNLOAD_REPORT_PATH, index=False)

display(download_df if not download_df.empty else pd.DataFrame(columns=['doc_id', 'status', 'source_path', 'error']))


## 7. PDF extraction

Converts source PDFs to markdown using `markitdown`. Existing outputs are reused unless `FORCE_REBUILD=True`.


In [ ]:
extraction_rows = extract_markdown(
    entries=entries,
    repo_root=REPO_ROOT,
    extracted_root=config.extracted_root,
    force_rebuild=FORCE_REBUILD,
)
extraction_df = pd.DataFrame(extraction_rows)
extraction_df.to_csv(EXTRACTION_REPORT_PATH, index=False)

display(extraction_df)


In [ ]:
sample_doc_ids = extraction_df.loc[extraction_df['status'].isin(['extracted', 'skipped_existing']), 'doc_id'].tolist()
if sample_doc_ids:
    sample_path = config.extracted_root / f"{sample_doc_ids[0]}.md"
    print('Sample extracted file:', sample_path)
    text = sample_path.read_text(encoding='utf-8')
    print('\n'.join(text.splitlines()[:40]))
else:
    print('No extracted markdown available yet.')



## 8. Cleaning / document preparation

- split markdown into lines
- remove empty lines
- collapse repeated whitespace
- keep source metadata attached


In [ ]:
cleaned_docs, clean_rows = prepare_cleaned_documents(
    entries=entries,
    repo_root=REPO_ROOT,
    extracted_root=config.extracted_root,
    cleaned_root=config.cleaned_root,
    force_rebuild=FORCE_REBUILD,
    normalize_whitespace=True,
)
clean_df = pd.DataFrame(clean_rows)
clean_df.to_csv(CLEAN_REPORT_PATH, index=False)
save_cleaned_documents_jsonl(cleaned_docs, CLEANED_DOCS_JSONL_PATH)

print('Cleaned documents:', len(cleaned_docs))
display(clean_df)


## 9. Corpus diagnostics

Basic corpus health checks before chunking.


In [ ]:
corpus_rows = [
    {
        'doc_id': doc.doc_id,
        'title': doc.title,
        'collection': doc.collection,
        'audience': doc.audience,
        'line_count': doc.line_count,
        'char_count': doc.char_count,
    }
    for doc in cleaned_docs
]
corpus_df = pd.DataFrame(corpus_rows).sort_values(['collection', 'doc_id']) if corpus_rows else pd.DataFrame()
if not corpus_df.empty:
    corpus_df.to_csv(CORPUS_DIAGNOSTICS_PATH, index=False)

print('Number of cleaned docs:', len(cleaned_docs))
display(corpus_df)


In [ ]:
preview_rows = []
for doc in cleaned_docs[:3]:
    preview_rows.append({
        'doc_id': doc.doc_id,
        'title': doc.title,
        'preview': '\n'.join(doc.content_lines[:8]),
    })

pd.DataFrame(preview_rows)



## 10. Chunking

Chunks are created with sliding windows and carry source metadata for attribution.


In [ ]:
chunks = build_chunks(cleaned_docs, size=CHUNK_SIZE, step=CHUNK_STEP)
save_chunks_jsonl(chunks, CHUNKS_JSONL_PATH)

chunk_counts_df = pd.DataFrame([
    {
        'doc_id': doc_id,
        'chunk_count': count,
    }
    for doc_id, count in sorted(pd.Series([c.doc_id for c in chunks]).value_counts().to_dict().items())
]) if chunks else pd.DataFrame(columns=['doc_id', 'chunk_count'])

print('Total chunks:', len(chunks))
display(chunk_counts_df)


## 11. Chunk diagnostics

Review chunk lengths and inspect sample chunks.


In [ ]:
chunk_diag_df = pd.DataFrame([
    {
        'chunk_id': c.chunk_id,
        'doc_id': c.doc_id,
        'collection': c.collection,
        'line_count': c.line_count,
        'char_count': c.char_count,
    }
    for c in chunks
])
if not chunk_diag_df.empty:
    chunk_diag_df.to_csv(CHUNK_DIAGNOSTICS_PATH, index=False)

print('Average char_count:', float(chunk_diag_df['char_count'].mean()) if not chunk_diag_df.empty else 0)
print('Shortest chunk chars:', int(chunk_diag_df['char_count'].min()) if not chunk_diag_df.empty else 0)
print('Longest chunk chars:', int(chunk_diag_df['char_count'].max()) if not chunk_diag_df.empty else 0)
chunk_diag_df.head()


In [ ]:
sample_chunks_df = pd.DataFrame([
    {
        'chunk_id': c.chunk_id,
        'source_title': c.source_title,
        'source_url': c.source_url,
        'preview': c.content[:500],
    }
    for c in chunks[:5]
])
sample_chunks_df


## 12. Index preparation

Create minsearch-ready rows and persist them for rebuild-on-demand.


In [ ]:
index_documents = prepare_index_documents(chunks)
save_index_documents(index_documents, INDEX_DOCUMENTS_JSONL_PATH)

index_input_df = pd.DataFrame(index_documents)
print('Index documents:', len(index_documents))
index_input_df.head()


## 13. Build index

`minsearch` index object is built in-memory. Persisted artifact is `index_documents.jsonl`, which is enough for fast rebuild in later notebooks.


In [ ]:
index = build_index(index_documents)
print('Index built in-memory with documents:', len(index_documents))


## 14. Search demo

MealMaster-relevant lexical retrieval smoke tests.


In [ ]:
demo_queries = [
    'healthy eating routine',
    'protein for older adults',
    'hydration tips',
    'calcium and vitamin d',
    'plant based substitutes',
    'mood and food',
]

search_rows = run_queries(index=index, queries=demo_queries, num_results=NUM_RESULTS)
search_df = pd.DataFrame(search_rows)
if not search_df.empty:
    search_df.to_csv(SEARCH_DEMO_REPORT_PATH, index=False)

search_display_df = search_df.reindex(columns=['query', 'rank', 'source_title', 'source_url', 'collection', 'snippet'])
search_display_df


## 15. Simple RAG demo

Retrieval-first behavior:
- if `OPENAI_API_KEY` is absent: retrieval-only + prompt preview
- if key is present: answer generation with usage metrics

Safety posture: source-grounded guidance only, no diagnosis.


In [ ]:
rag_query = 'What are healthy eating patterns for older adults?'
rag_result = rag_answer(index=index, query=rag_query, model=LLM_MODEL, num_results=NUM_RESULTS)
print('Mode:', rag_result['mode'])
print('Question:', rag_result['question'])
print('Usage:', rag_result['usage'])

if rag_result['mode'] == 'llm':
    print('\nAnswer:\n', rag_result['answer'])
else:
    print('\nNo API key detected; showing prompt preview and retrieval references.')
    print(rag_result['prompt_preview'][:1500], '...')

pd.DataFrame(rag_result['references'])



## 16. Structured output demo

Optional typed response using a small Pydantic schema.


In [ ]:
structured_result = rag_answer_structured(index=index, query='How can I improve hydration?', model=LLM_MODEL, num_results=NUM_RESULTS)
print('Mode:', structured_result['mode'])
print('Usage:', structured_result['usage'])
print(json.dumps(structured_result['response'], indent=2))


## 17. Save artifacts summary

This is the handoff inventory for later notebooks.


In [ ]:
artifact_rows = []
for path in sorted(config.data_root.rglob('*')):
    if path.is_file():
        artifact_rows.append(
            {
                'path': str(path.relative_to(REPO_ROOT)),
                'size_bytes': path.stat().st_size,
            }
        )

artifact_df = pd.DataFrame(artifact_rows)
artifact_df.to_csv(ARTIFACT_SUMMARY_PATH, index=False)
artifact_df.head(50)


## 18. Next steps

Use these artifacts in later course notebooks to add:
1. stronger retrieval tuning (reranking, filters, hybrid)
2. stricter structured outputs and routing
3. agent tool wrappers over retrieval/index refresh
4. offline eval sets and automated scoring
5. monitoring for drift, failures, and retrieval quality
